# 01 · Data preparation

Milestone 1 of the profit-aware campaign optimizer. This notebook only **calls** the `pacopt` package (`data.py`, `features.py`), so all of the logic is shared with the pipeline and the Streamlit app.

**Steps:** load and validate → clean categoricals → RFM-inspired features → eligibility flags → leakage guard → stratified split → 10,000-customer campaign universe.

In [1]:
import pandas as pd
import plotly.express as px
import plotly.io as pio
from pacopt import config as cfg, data, features

pio.renderers.default = 'notebook_connected'
pd.set_option('display.width', 140)

## 1. Load and validate the raw data
`load_raw()` checks the 17-column schema, the 45,211 row count, that there are no nulls and the ~11.7% subscription rate.

In [2]:
raw = data.load_raw()
print(raw.shape, '| SHA-256 OK:', data.sha256(cfg.RAW_CSV) == cfg.RAW_CSV_SHA256)
raw.head()

(45211, 17) | SHA-256 OK: True


,age,job,marital,education,default,balance,housing,loan,contact,day,month,duration,campaign,pdays,previous,poutcome,y
0,58,management,married,tertiary,no,2143,yes,no,unknown,5,may,261,1,-1,0,unknown,no
1,44,technician,single,secondary,no,29,yes,no,unknown,5,may,151,1,-1,0,unknown,no
2,33,entrepreneur,married,secondary,no,2,yes,yes,unknown,5,may,76,1,-1,0,unknown,no
3,47,blue-collar,married,unknown,no,1506,yes,no,unknown,5,may,92,1,-1,0,unknown,no
4,33,unknown,single,unknown,no,1,no,no,unknown,5,may,198,1,-1,0,unknown,no


In [3]:
raw['y'].value_counts(normalize=True).rename('share').to_frame()

,share
y,
no,0.883015
yes,0.116985


## 2. Clean categorical features
Values are stripped and lower-cased, `admin.` becomes `admin`, binaries become 0/1, and **`unknown` stays as its own category** because it carries information (e.g. no recorded phone channel).

In [4]:
clean = data.clean(raw)
{c: list(clean[c].cat.categories) for c in ['job', 'contact', 'poutcome']}

{'job': ['admin',
  'blue-collar',
  'entrepreneur',
  'housemaid',
  'management',
  'retired',
  'self-employed',
  'services',
  'student',
  'technician',
  'unemployed',
  'unknown'],
 'contact': ['cellular', 'telephone', 'unknown'],
 'poutcome': ['failure', 'other', 'success', 'unknown']}

## 3. Leakage check: why `duration`, `campaign`, `day` and `month` are excluded
These columns describe what happened **during** the campaign. The table shows how strongly `campaign` (contacts in this campaign) relates to the outcome. Customers who say yes stop being called, so a low count is partly a *result* of subscribing.

In [5]:
camp = clean.assign(campaign_band=pd.cut(clean['campaign'], [0, 1, 2, 3, 5, 100],
                                         labels=['1', '2', '3', '4-5', '6+']))
camp.groupby('campaign_band', observed=True)['y'].agg(['mean', 'size']).rename(columns={'mean': 'conversion'})

,conversion,size
campaign_band,,
1,0.145976,17544
2,0.112035,12505
3,0.111936,5521
4-5,0.086266,5286
6+,0.058094,4355


In [6]:
month_order = ['jan','feb','mar','apr','may','jun','jul','aug','sep','oct','nov','dec']
m = clean.groupby('month', observed=True)['y'].agg(['mean', 'size']).reindex(month_order)
m.rename(columns={'mean': 'conversion', 'size': 'calls'})  # small Mar/Sep/Oct/Dec waves convert ~45-50%: calling schedule, not a customer trait

,conversion,calls
month,,
jan,0.101212,1403
feb,0.166478,2649
mar,0.519916,477
apr,0.196794,2932
may,0.067195,13766
jun,0.102228,5341
jul,0.090935,6895
aug,0.110133,6247
sep,0.464594,579


In [7]:
print('Model features:', features.MODEL_FEATURES)
features.assert_no_leakage(features.MODEL_FEATURES)
print('Leakage guard passed - forbidden:', sorted(cfg.FORBIDDEN_FEATURES))

Model features: ['age', 'balance_slog', 'frequency', 'never_contacted', 'prior_success', 'housing', 'loan', 'default', 'job', 'marital', 'education', 'contact', 'poutcome', 'age_band', 'recency_bucket']
Leakage guard passed - forbidden: ['campaign', 'day', 'duration', 'month', 'y']


## 4. RFM-inspired features and eligibility flags
| RFM | Feature | Note |
|---|---|---|
| Recency | `never_contacted`, `recency_days`, `recency_bucket` | from `pdays` (-1 = never) |
| Frequency | `frequency` = `previous` (capped at 10), `prior_success` | prior campaigns only |
| Monetary proxy | `balance`, `balance_slog`, `balance_quintile` | a value indicator, **not revenue** |

In [8]:
feats, parts = features.prepare(raw)
feats[['customer_id','age','age_band','balance','balance_slog','balance_quintile',
       'pdays','never_contacted','recency_bucket','frequency','prior_success','eligible']].head()

,customer_id,age,age_band,balance,balance_slog,balance_quintile,pdays,never_contacted,recency_bucket,frequency,prior_success,eligible
0,C100001,58,45-59,2143,7.670429,5,-1,1,never,0,0,1
1,C100002,44,30-44,29,3.401197,2,-1,1,never,0,0,1
2,C100003,33,30-44,2,1.098612,1,-1,1,never,0,0,1
3,C100004,47,45-59,1506,7.317876,4,-1,1,never,0,0,1
4,C100005,33,30-44,1,0.693147,1,-1,1,never,0,0,1


### Conversion by previous-campaign outcome (strongest pre-campaign signal)

In [9]:
po = feats.groupby('poutcome', observed=True)['y'].agg(['mean', 'size']).sort_values('mean')
fig = px.bar(po.reset_index(), x='poutcome', y='mean', text_auto='.1%',
             labels={'mean': 'Conversion rate', 'poutcome': 'Previous outcome'},
             title='Conversion rate by previous campaign outcome')
fig.update_traces(marker_color='#4F97FB', marker_cornerradius=6)
fig.update_layout(plot_bgcolor='white', yaxis_tickformat='.0%', height=380)
fig.show()
po

,mean,size
poutcome,,
unknown,0.091615,36959
failure,0.126097,4901
other,0.166848,1840
success,0.647253,1511


### Conversion by recency bucket

In [10]:
rec_order = ['never', '<=90d', '91-180d', '181-365d', '>365d']
feats.groupby('recency_bucket', observed=True)['y'].agg(['mean', 'size']).reindex(rec_order)

,mean,size
recency_bucket,,
never,0.091573,36954
<=90d,0.349582,718
91-180d,0.274194,2480
181-365d,0.177989,4416
>365d,0.292379,643


### Balance distribution (monetary proxy)
Balance is heavily right-skewed with a negative tail, so the model uses a signed log transform. Negative balances are excluded from targeting (eligibility rule).

In [11]:
print(feats['balance'].describe(percentiles=[.05, .25, .5, .75, .95, .99]).round(0))
fig = px.histogram(feats, x='balance_slog', nbins=60, color='y',
                   labels={'balance_slog': 'sign(balance) * log(1+|balance|)', 'y': 'Subscribed'},
                   title='Signed-log balance by outcome', barmode='overlay', opacity=0.7)
fig.update_layout(plot_bgcolor='white', height=380)
fig.show()

count     45211.0
mean       1362.0
std        3045.0
min       -8019.0
5%         -172.0
25%          72.0
50%         448.0
75%        1428.0
95%        5768.0
99%       13165.0
max      102127.0
Name: balance, dtype: float64


In [12]:
feats.groupby('balance_quintile')['y'].agg(['mean', 'size']).rename(columns={'mean': 'conversion'})

,conversion,size
balance_quintile,,
1,0.069446,9043
2,0.099757,9042
3,0.117341,9042
4,0.134705,9042
5,0.163681,9042


## 5. Eligibility (responsible-customer rules, D-007)
Hard exclusions: credit default, negative balance, contact fatigue (`previous > 10`). Customers aged 75+ are **flagged** (capped in optimisation), not excluded.

In [13]:
elig_cols = ['elig_default', 'elig_negative_balance', 'elig_fatigue', 'flag_age_75plus']
summary = pd.DataFrame({'customers': feats[elig_cols].sum(),
                        'share': feats[elig_cols].mean().round(4)})
summary.loc['eligible (all rules)'] = [feats['eligible'].sum(), round(feats['eligible'].mean(), 4)]
summary

,customers,share
elig_default,815,0.0180
elig_negative_balance,3766,0.0833
elig_fatigue,294,0.0065
flag_age_75plus,300,0.0066
eligible (all rules),40802,0.9025


## 6. Train/test split and the 10,000-customer campaign universe
Stratified 70/30 split. The campaign universe is drawn **only from the test set**, so every propensity score used for decisions is out-of-sample (D-014).

In [14]:
split_summary = pd.DataFrame({
    'rows': [len(parts.train), len(parts.test), len(parts.universe)],
    'positive_rate': [parts.train['y'].mean(), parts.test['y'].mean(), parts.universe['y'].mean()],
    'eligible_share': [parts.train['eligible'].mean(), parts.test['eligible'].mean(),
                       parts.universe['eligible'].mean()],
}, index=['train', 'test', 'universe']).round(4)
assert len(parts.universe) == 10_000
assert set(parts.universe['customer_id']) <= set(parts.test['customer_id'])
split_summary

,rows,positive_rate,eligible_share
train,31647,0.117,0.9030
test,13564,0.117,0.9014
universe,10000,0.117,0.9041


In [15]:
parts.universe['contact'].value_counts().rename('customers by contact channel').to_frame()

,customers by contact channel
contact,
cellular,6489
unknown,2881
telephone,630


## 7. Save processed data

In [16]:
features.save_processed(feats, parts)
sorted(p.name for p in cfg.PROCESSED_DIR.glob('*.parquet'))

['customers_features.parquet',
 'test.parquet',
 'train.parquet',
 'universe.parquet']

**Milestone 1 output:** a clean feature table with RFM-inspired features and eligibility flags, a leakage-guarded feature allow-list, and stratified train/test sets plus a 10,000-customer out-of-sample campaign universe in `data/processed/`.